# Munich NDVI map
This notebook creates an NDVI map for the Munich city center within a 15 km radius and displays it inline in the notebook.

In [1]:
import ee
from IPython.display import Image, display

# Initialize Earth Engine with the configured project
try:
    ee.Initialize(project='ee-simonmarggraf')
except Exception:
    ee.Authenticate(auth_mode='localhost')
    ee.Initialize(project='ee-simonmarggraf')

# Munich city center coordinates and 15 km radius ROI
munich_center = ee.Geometry.Point([11.576124, 48.137154])
roi = munich_center.buffer(15000)

# Sentinel-2 surface reflectance composite for a recent growing season
sentinel2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2025-05-01', '2025-09-30')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .median()
)

# Compute NDVI
ndvi = sentinel2.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndvi_clipped = ndvi.clip(roi)

# Display settings
ndvi_vis = {
    'min': -0.5,
    'max': 0.8,
    'palette': [
        '#d73027', '#f46d43', '#fdae61', '#fee08b', '#ffffbf',
        '#a6d96a', '#66bd63', '#1a9850'
    ],
}

# Build a static NDVI image that should display reliably in VS Code notebooks.
ndvi_thumb = ndvi_clipped.visualize(**ndvi_vis).getThumbURL({
    'region': roi.getInfo()['coordinates'],
    'dimensions': 900,
    'format': 'png',
})

# Show the actual NDVI image inline
Image(url=ndvi_thumb)

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python
